<div align="center">

# 🚛 FleetLogix · Quick Start

**Notebook 0 · Punto de entrada del proyecto**

*Henry Data Science · Proyecto 2 · Dody Dueñas*

</div>

---

## 🎯 Propósito

Este notebook es la **puerta de entrada** al proyecto FleetLogix. En menos de **5 minutos** valida que tu entorno esté listo para correr los 4 Avances y lanzar el dashboard Streamlit.

## 📚 Índice del proyecto

| Notebook / Entregable | Tema | Tiempo |
|---|---|---|
| **0 · Quick Start** *(este)* | Validación de entorno + setup express | 5 min |
| **1 · Data Generation** | Diseño del schema + Faker para 535k filas | 30 min |
| **2 · SQL Analysis** | 12 queries analíticos + 11 índices | 45 min |
| **3 · Data Warehouse** | Modelo dimensional Snowflake (star schema) | 30 min |
| **4 · Cloud Architecture** | Diagrama AWS · ETL · Lambda · S3 | 30 min |
| **Dashboard Streamlit** | `dashboard_streamlit/streamlit_app.py` | — |

## 🗂 Estructura del repositorio

```
Proyecto2Dody/
├── docker-compose.yml           # PostgreSQL + PostgREST + Swagger
├── .env                         # Variables de entorno
├── requirements.txt             # Dependencias Python
├── sql/                         # Scripts SQL (schema, queries, índices)
├── scripts/                     # Generación de datos, ETL, AWS simulator
├── notebooks/                   # 4 Avances + este Quick Start
├── dashboard_streamlit/         # App Streamlit multipágina
│   ├── streamlit_app.py         # Página 1: Resumen Ejecutivo
│   ├── pages/                   # Flota · Conductores · Rutas · Combustible
│   ├── utils/                   # db.py (engine) · styling.py (tema)
│   └── .streamlit/config.toml   # Tema oscuro corporativo
├── tests/                       # Tests unitarios
└── README.md
```

---

## 1. ⚙️ Validación de entorno

Verifica versiones mínimas requeridas.

In [1]:
import sys, platform, subprocess

REQUIRED_PYTHON = (3, 10)

print(f"Python   : {sys.version.split()[0]}  ({platform.platform()})")
ok_py = sys.version_info >= REQUIRED_PYTHON
print(f"  + Python >= {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}" if ok_py else f"  X Necesitas Python >= {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}")

for tool in ("docker", "psql"):
    try:
        out = subprocess.check_output([tool, "--version"], text=True, stderr=subprocess.STDOUT).strip()
        print(f"{tool:9s}: {out}")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print(f"{tool:9s}: no encontrado en PATH")

Python   : 3.11.9  (Windows-10-10.0.19045-SP0)
  + Python >= 3.10
docker   : Docker version 29.2.1, build a5c7197
psql     : no encontrado en PATH


## 2. 📦 Dependencias Python

In [2]:
import importlib

REQUIRED = {
    "psycopg2":   "PostgreSQL adapter",
    "sqlalchemy": "ORM/Engine",
    "pandas":     "DataFrames",
    "faker":      "Datos sinteticos",
    "matplotlib": "Visualizacion",
    "seaborn":    "Visualizacion estadistica",
    "streamlit":  "Dashboard web",
    "plotly":     "Graficos interactivos",
}

missing = []
for pkg, desc in REQUIRED.items():
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "?")
        print(f"  + {pkg:14s} {ver:12s}  · {desc}")
    except ImportError:
        print(f"  X {pkg:14s} {'FALTA':12s}  · {desc}")
        missing.append(pkg)

if missing:
    print(f"\nInstala las que faltan con:\n  pip install {' '.join(missing)}")
else:
    print("\nTodas las dependencias estan instaladas.")

  + psycopg2       2.9.9 (dt dec pq3 ext lo64)  · PostgreSQL adapter


  + sqlalchemy     2.0.28        · ORM/Engine


  + pandas         2.2.1         · DataFrames
  + faker          ?             · Datos sinteticos


  + matplotlib     3.10.7        · Visualizacion


  + seaborn        0.13.2        · Visualizacion estadistica


  + streamlit      1.54.0        · Dashboard web
  + plotly         6.5.0         · Graficos interactivos

Todas las dependencias estan instaladas.


## 3. 🐘 Conexión a PostgreSQL

In [3]:
import os
import sys
from pathlib import Path

# Carga .env desde la raiz del proyecto
try:
    from dotenv import load_dotenv
    env_path = Path("__file__").resolve().parents[1] / ".env"
    # Buscar .env en directorios padre
    for p in [Path("."), Path(".."), Path("../..")]:
        candidate = p / ".env"
        if candidate.exists():
            load_dotenv(candidate)
            print(f"  .env cargado desde: {candidate.resolve()}")
            break
    else:
        print("  .env no encontrado, usando valores por defecto")
except ImportError:
    print("  python-dotenv no instalado, usando valores por defecto")

DSN = dict(
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", 5432)),
    dbname=os.getenv("DB_NAME", "fleetlogix_db"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", "your_password"),
)

print(f"  Conectando a {DSN['user']}@{DSN['host']}:{DSN['port']}/{DSN['dbname']}")

try:
    import psycopg2
    with psycopg2.connect(**DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            ver = cur.fetchone()[0].split(',')[0]
            print(f"  + Conectado: {ver}")
            cur.execute(
                """SELECT table_name
                   FROM information_schema.tables
                   WHERE table_schema = 'public'
                   ORDER BY table_name;"""
            )
            tables = [r[0] for r in cur.fetchall()]
            print(f"  + Tablas/Vistas en public ({len(tables)}): {', '.join(tables[:10]) or '- vacio -'} ...")
except Exception as exc:
    print(f"  X No se pudo conectar: {exc}")
    print("  Asegurate de que PostgreSQL este corriendo y las credenciales sean correctas.")

  .env cargado desde: C:\Users\DODY DUEÑAS\Documents\Poryecto2Henry\Proyecto2Dody\.env
  Conectando a postgres@localhost:5432/fleetlogix_db
  + Conectado: PostgreSQL 18.3 on x86_64-windows
  + Tablas/Vistas en public (10): customers, deliveries, dim_date, drivers, maintenance, routes, system_logs, trips, v_dim_date, vehicles ...


## 4. 📊 Validación de las vistas del dashboard

In [4]:
import psycopg2
import pandas as pd

VIEWS = [
    "v_kpi_executive", "v_deliveries_timeseries", "v_vehicle_performance",
    "v_driver_performance", "v_route_traffic", "v_maintenance_alerts",
    "v_fuel_efficiency",
]

rows = []
try:
    with psycopg2.connect(**DSN) as conn, conn.cursor() as cur:
        for v in VIEWS:
            try:
                cur.execute(f'SELECT COUNT(*) FROM "{v}"')
                rows.append({"vista": v, "filas": cur.fetchone()[0], "estado": "OK"})
            except Exception as exc:
                rows.append({"vista": v, "filas": 0, "estado": f"FALTA ({type(exc).__name__})"})
                conn.rollback()
except Exception as exc:
    print(f"Error de conexion: {exc}")
    rows = [{"vista": v, "filas": 0, "estado": "SIN CONEXION"} for v in VIEWS]

df = pd.DataFrame(rows)
print(df.to_string(index=False))
ok_count = (df['estado'] == 'OK').sum()
print(f"\n{ok_count}/{len(VIEWS)} vistas disponibles")

                  vista  filas                 estado
        v_kpi_executive      0 FALTA (UndefinedTable)
v_deliveries_timeseries      0 FALTA (UndefinedTable)
  v_vehicle_performance      0 FALTA (UndefinedTable)
   v_driver_performance      0 FALTA (UndefinedTable)
        v_route_traffic      0 FALTA (UndefinedTable)
   v_maintenance_alerts      0 FALTA (UndefinedTable)
      v_fuel_efficiency      0 FALTA (UndefinedTable)

0/7 vistas disponibles


## 5. 🎯 Próximos pasos

Si todos los chequeos están en verde:

1. Continúa con **`Avance_1_DataGeneration.ipynb`** para profundizar en el diseño del schema.
2. Lanza el dashboard:
   ```bash
   cd dashboard_streamlit
   streamlit run streamlit_app.py
   ```
   Y navega a `http://localhost:8501`
3. Para ejecutar la simulacion AWS:
   ```bash
   python -X utf8 scripts/aws_local_simulator.py
   ```

---

<div align="center">

**FleetLogix · 2026** · *Notebook generado para uso academico*  
dodydurema67@gmail.com

</div>